In [29]:
import pandas as pd
import numpy as np
import requests

### Считаем json файл который мы получили в ex02

In [30]:
df = pd.read_json('../data/auto.json', orient='records')
pd.options.display.float_format = '{:.2f}'.format

In [31]:
df.head()

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus


### Обогатим датафрейм используя выборку из этого датафрейма

In [32]:
sample = df.sample(n=200, random_state=21)
sample['Refund'] = np.random.choice(df['Refund'], size=len(sample))
sample['Fines'] = np.random.choice(df['Fines'], size=len(sample))

In [33]:
concat_rows = pd.concat([df, sample], ignore_index=True)

In [34]:
concat_rows.head()

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus


### Обогатим наш датафрейм новым стобцом с сгенерированными данными

In [35]:
np.random.seed(21)
years = np.random.randint(1980, 2020, size=len(concat_rows))
years = pd.Series(years, name = "Year")
fines = pd.concat([concat_rows, years], axis = 1)

In [36]:
fines.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014


In [37]:
surnames = pd.read_json('../../datasets/surname.json', orient='records')
columns = surnames.iloc[0]
surnames = surnames[1:]
surnames.columns = columns
surnames

,NAME,COUNT,RANK
1,ADAMS,427865,42
2,ALLEN,482607,33
3,ALVAREZ,233983,92
4,ANDERSON,784404,15
5,BAILEY,277845,72
...,...,...,...
96,WILLIAMS,1625252,3
97,WILSON,801882,14
98,WOOD,250715,84
99,WRIGHT,458980,35


In [38]:
car_numbers = fines['CarNumber'].drop_duplicates()
car_numbers = car_numbers.to_frame().reset_index(drop=True)
car_numbers

,CarNumber
0,Y163O8161RUS
1,E432XX77RUS
2,7184TT36RUS
3,X582HE161RUS
4,92918M178RUS
...,...
526,O136HO197RUS
527,O22097197RUS
528,M0309X197RUS
529,O673E8197RUS


In [39]:
surname = surnames['NAME'].sample(n=len(car_numbers), random_state=21, replace=True)
surname = surname.to_frame(name='SURNAME').reset_index(drop=True)
surname

,SURNAME
0,RICHARDSON
1,ROSS
2,MORGAN
3,BAILEY
4,LOPEZ
...,...
526,CAMPBELL
527,HALL
528,BAKER
529,DIAZ


In [40]:
owners = pd.concat([car_numbers, surname], axis=1)
owners

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
526,O136HO197RUS,CAMPBELL
527,O22097197RUS,HALL
528,M0309X197RUS,BAKER
529,O673E8197RUS,DIAZ


In [41]:
my_samples = [['12345RUS', 1, 1000.00, 'Volkswagen', 'Passat', 2005],
              ['51234RUS', 1, 1500.00, 'Volkswagen', 'Passat', 2005],
              ['45123RUS', 1, 2000.00, 'Volkswagen', 'Passat', 2005],
              ['34512RUS', 1, 2500.00, 'Volkswagen', 'Passat', 2005],
              ['23451RUS', 1, 3000.00, 'Volkswagen', 'Passat', 2005]]
my_samples = pd.DataFrame(my_samples, columns=fines.columns)
fines = pd.concat([fines, my_samples])

In [42]:
owners.drop(owners.tail(20).index, inplace=True)
my_owners = [['123RUS', 'NODAR'],
             ['312RUS', 'STEPAN'],
             ['231RUS', 'MALIK']]
my_owners = pd.DataFrame(my_owners, columns=owners.columns)
owners = pd.concat([owners, my_owners])
owners

,CarNumber,SURNAME
0,Y163O8161RUS,RICHARDSON
1,E432XX77RUS,ROSS
2,7184TT36RUS,MORGAN
3,X582HE161RUS,BAILEY
4,92918M178RUS,LOPEZ
...,...,...
509,O50197197RUS,WRIGHT
510,7608EE777RUS,HILL
0,123RUS,NODAR
1,312RUS,STEPAN


In [43]:
fines_1 = pd.merge(left=fines,
                   right=owners,
                   how='inner',
                   on='CarNumber')
fines_1

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ
...,...,...,...,...,...,...,...
894,8182XX154RUS,1,1400.00,Ford,Focus,1981,SMITH
895,X796TH96RUS,1,800.00,Ford,Focus,1992,WATSON
896,T011MY163RUS,1,3000.00,Ford,Focus,2007,SANDERS
897,T341CC96RUS,1,1000.00,Volkswagen,Passat,2005,PEREZ


In [44]:
fines_2 = pd.merge(left=fines,
                   right=owners,
                   how='outer',
                   on='CarNumber')
fines_2

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,12345RUS,1.00,1000.00,Volkswagen,Passat,2005.00,NaN
1,123RUS,NaN,NaN,NaN,NaN,NaN,NODAR
2,231RUS,NaN,NaN,NaN,NaN,NaN,MALIK
3,23451RUS,1.00,3000.00,Volkswagen,Passat,2005.00,NaN
4,312RUS,NaN,NaN,NaN,NaN,NaN,STEPAN
...,...,...,...,...,...,...,...
928,Y973O8197RUS,2.00,8594.59,Ford,Focus,2005.00,YOUNG
929,Y973O8197RUS,1.00,34800.00,Ford,Focus,2003.00,YOUNG
930,Y973O8197RUS,1.00,69600.00,Ford,Focus,2017.00,YOUNG
931,Y973O8197RUS,1.00,1600.00,Ford,Focus,1987.00,YOUNG


In [45]:
fines_3 = pd.merge(left=fines,
                   right=owners,
                   how='left',
                   on='CarNumber')
fines_3

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ
...,...,...,...,...,...,...,...
925,12345RUS,1,1000.00,Volkswagen,Passat,2005,NaN
926,51234RUS,1,1500.00,Volkswagen,Passat,2005,NaN
927,45123RUS,1,2000.00,Volkswagen,Passat,2005,NaN
928,34512RUS,1,2500.00,Volkswagen,Passat,2005,NaN


In [46]:
fines_4 = pd.merge(left=fines,
                   right=owners,
                   how='right',
                   on='CarNumber')
fines_4

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2.00,3200.00,Ford,Focus,1989.00,RICHARDSON
1,Y163O8161RUS,2.00,1600.00,Ford,Focus,1980.00,RICHARDSON
2,E432XX77RUS,1.00,6500.00,Toyota,Camry,1995.00,ROSS
3,E432XX77RUS,2.00,13000.00,Toyota,Camry,2018.00,ROSS
4,7184TT36RUS,1.00,2100.00,Ford,Focus,1984.00,MORGAN
...,...,...,...,...,...,...,...
897,7608EE777RUS,1.00,4000.00,Skoda,Octavia,2000.00,HILL
898,7608EE777RUS,2.00,8594.59,Skoda,Octavia,1991.00,HILL
899,123RUS,NaN,NaN,NaN,NaN,NaN,NODAR
900,312RUS,NaN,NaN,NaN,NaN,NaN,STEPAN


### Создайте сводную таблицу из фрейма данных о штрафах, она должна выглядеть так (значения — это суммы штрафов), но со всеми годами

In [47]:
pivot_table = fines.pivot_table(values='Fines',
                                index=['Make', 'Model'],
                                columns='Year',
                                aggfunc='sum')
pivot_table

Year                    1980      1981      1982     1983      1984      1985  \
Make       Model                                                                
Ford       Focus    93389.17 414089.17 178183.76 67000.00 124194.59 134483.76   
           Mondeo        NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia 105500.00       NaN  15494.59 11594.59       NaN  10294.59   
Toyota     Camry    12000.00   8594.59       NaN  7200.00       NaN       NaN   
           Corolla       NaN       NaN   2000.00      NaN       NaN       NaN   
Volkswagen Golf     30900.00       NaN       NaN  8594.59    300.00  24000.00   
           Jetta         NaN       NaN       NaN      NaN       NaN       NaN   
           Passat        NaN   1600.00       NaN  3200.00  10000.00   5000.00   
           Touareg       NaN       NaN       NaN      NaN       NaN   5800.00   

Year                   1986     1987      1988     1989  ...      2010  \
Make       Model                                         ...             
Ford       Focus   88894.59 83900.00 169094.59 70300.00  ... 122183.76   
           Mondeo       NaN      NaN       NaN  8600.00  ...       NaN   
Skoda      Octavia   600.00  5200.00  12000.00 91400.00  ...   3100.00   
Toyota     Camry        NaN      NaN       NaN 22400.00  ...       NaN   
           Corolla      NaN 16000.00       NaN  4000.00  ...  24000.00   
Volkswagen Golf         NaN 20000.00       NaN  5800.00  ...       NaN   
           Jetta        NaN      NaN       NaN      NaN  ...       NaN   
           Passat  15000.00 12300.00       NaN      NaN  ...   2800.00   
           Touareg      NaN      NaN       NaN      NaN  ...   6300.00   

Year                   2011     2012      2013      2014      2015     2016  \
Make       Model                                                              
Ford       Focus   97083.76 92900.00 149489.17 116894.59 248300.00 82794.59   
           Mondeo       NaN 34400.00       NaN       NaN       NaN 46200.00   
Skoda      Octavia   500.00   500.00  12594.59    300.00  46394.59   300.00   
Toyota     Camry        NaN  8594.59       NaN  11000.00       NaN      NaN   
           Corolla  8594.59   400.00       NaN       NaN       NaN  1300.00   
Volkswagen Golf      300.00      NaN   7400.00       NaN   2300.00      NaN   
           Jetta        NaN      NaN       NaN       NaN       NaN      NaN   
           Passat       NaN      NaN       NaN       NaN    600.00  2100.00   
           Touareg      NaN      NaN       NaN   1300.00    500.00      NaN   

Year                    2017      2018     2019  
Make       Model                                 
Ford       Focus   313400.00 269694.59 93994.59  
           Mondeo        NaN       NaN      NaN  
Skoda      Octavia   2500.00 156200.00  9500.00  
Toyota     Camry         NaN  35400.00 18100.00  
           Corolla   9600.00   5000.00      NaN  
Volkswagen Golf          NaN  30300.00      NaN  
           Jetta         NaN       NaN      NaN  
           Passat        NaN       NaN      NaN  
           Touareg       NaN       NaN      NaN  

[9 rows x 40 columns]

In [50]:
fines.to_csv('../data/fines.csv', index=False)
owners.to_csv('../data/owners.csv', index=False)